# 02_Transformacion_Silver

Transformación de las cuatro tablas Bronze hacia Silver.

Objetivos de esta capa:

- eliminar columnas técnicas del CSV que no aportan al modelo (`_c0`);
- tipar fechas y campos numéricos;
- convertir coordenadas con coma decimal a `DOUBLE`;
- limpiar cadenas vacías y espacios;
- estandarizar algunas categorías sin alterar el grano natural;
- conservar metadatos de trazabilidad;
- validar conteos, claves e integridad referencial.

Las transformaciones se basan en la exploración previa de las tablas Bronze.


## 1. Configuración


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text(
    "catalog_name",
    "accidentalidad_databricks",
    "Catalog del proyecto"
)

PROY_CATALOG = dbutils.widgets.get("catalog_name")
PROY_SCHEMA_BRONZE = "bronze"
PROY_SCHEMA_SILVER = "silver"

print(f"Catalog: {PROY_CATALOG}")
print(f"Bronze:  {PROY_SCHEMA_BRONZE}")
print(f"Silver:  {PROY_SCHEMA_SILVER}")


## 2. Funciones auxiliares

Se usan funciones tolerantes para que un valor inesperado no detenga todo el pipeline.

- `limpiar_texto`: elimina espacios y convierte cadenas vacías en `NULL`.
- `decimal_con_coma`: transforma valores como `-74,842098` en `-74.842098`.
- `fecha_ddmmyyyy`: convierte fechas de accidente en formato `dd/MM/yyyy`.
- `fecha_iso`: convierte fechas en formato `yyyy-MM-dd`.
- `fecha_nacimiento`: soporta tanto `yyyy-MM-dd` como `yyyy-MM-dd HH:mm:ss`.


In [ ]:
def limpiar_texto(columna):
    valor = F.trim(F.col(columna))
    return F.when(valor == "", F.lit(None)).otherwise(valor)


def decimal_con_coma(columna):
    return F.expr(
        f"try_cast(replace(trim(`{columna}`), ',', '.') AS DOUBLE)"
    )


def fecha_ddmmyyyy(columna):
    return F.expr(
        f"try_to_date(`{columna}`, 'dd/MM/yyyy')"
    )


def fecha_iso(columna):
    return F.expr(
        f"try_to_date(`{columna}`, 'yyyy-MM-dd')"
    )


def fecha_nacimiento(columna):
    return F.coalesce(
        F.to_date(
            F.expr(
                f"try_to_timestamp(`{columna}`, 'yyyy-MM-dd HH:mm:ss')"
            )
        ),
        F.expr(
            f"try_to_date(`{columna}`, 'yyyy-MM-dd')"
        ),
        F.expr(
            f"try_to_date(`{columna}`, 'dd/MM/yyyy')"
        )
    )


## 3. Transformación de accidentes

Grano: **una fila por accidente (`croquis`)**.

Decisiones principales:

- `_c0` se elimina porque corresponde al índice exportado del CSV.
- `fecha_accidente` pasa a `DATE`.
- año, mes y día pasan a `INT`.
- heridos y muertos pasan a `INT`, conservando `NULL` cuando la fuente no reporta valor.
- coordenadas pasan a `DOUBLE`.
- los campos categóricos principales se estandarizan en mayúsculas.


In [ ]:
df_accidentes_bronze = spark.table(
    f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.accidentes"
)

df_accidentes_silver = (
    df_accidentes_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("nro_heridos", F.expr("CAST(try_cast(nro_heridos AS DOUBLE) AS INT)"))
    .withColumn("numero_muertos", F.expr("CAST(try_cast(numero_muertos AS DOUBLE) AS INT)"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("gravedad", F.upper(limpiar_texto("gravedad")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("localidad", F.upper(limpiar_texto("localidad")))
    .withColumn("sitio_exacto", limpiar_texto("sitio_exacto"))
    .withColumn("placa_agente", F.upper(limpiar_texto("placa_agente")))
    .withColumn("apellidos", limpiar_texto("apellidos"))
    .withColumn("nombres", limpiar_texto("nombres"))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_accidentes_silver.limit(10))
df_accidentes_silver.printSchema()


## 4. Transformación de causas

Grano: **una fila por accidente (`croquis`)**.

Además del tipado de la fecha, se corrige el carácter `¤` observado en textos como `Desobedecer se¤ales`, interpretándolo como `ñ`.


In [ ]:
df_causas_bronze = spark.table(
    f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.causas"
)

df_causas_silver = (
    df_causas_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn(
        "hipotesis_causa",
        F.regexp_replace(limpiar_texto("hipotesis_causa"), "¤", "ñ")
    )
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_causas_silver.limit(10))
df_causas_silver.printSchema()


## 5. Transformación de vehículos

Grano: **una fila por vehículo involucrado**.

La exploración mostró dos formatos de fecha diferentes:

- `fecha_accidente`: `dd/MM/yyyy`
- `fecha_radica`: `yyyy-MM-dd`

Por ello cada campo se convierte con su formato correspondiente.


In [ ]:
df_vehiculos_bronze = spark.table(
    f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.vehiculos"
)

df_vehiculos_silver = (
    df_vehiculos_bronze
    .drop("_c0")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("fecha_radica", fecha_iso("fecha_radica"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("direccion", limpiar_texto("direccion"))
    .withColumn("placa", F.upper(limpiar_texto("placa")))
    .withColumn("clase", F.upper(limpiar_texto("clase")))
    .withColumn("servicio", F.upper(limpiar_texto("servicio")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_vehiculos_silver.limit(10))
df_vehiculos_silver.printSchema()


## 6. Transformación de víctimas

Grano: **una fila por víctima**.

La exploración permitió identificar que:

- `clase` contiene la clase del accidente (`Choque`, `Atropello`, etc.);
- `clase_1` contiene la clase del vehículo (`MOTOCICLETA`, `AUTOMOVIL`, etc.).

Por claridad semántica se renombran a `clase_accidente` y `clase_vehiculo`.

`fecha_nacimiento` aparece en formatos tipo `yyyy-MM-dd HH:mm:ss` y `yyyy-MM-dd`, por lo que se convierte de manera tolerante.


In [ ]:
df_victimas_bronze = spark.table(
    f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.victimas"
)

df_victimas_silver = (
    df_victimas_bronze
    .drop("_c0")
    .withColumnRenamed("clase", "clase_accidente")
    .withColumnRenamed("clase_1", "clase_vehiculo")
    .withColumn("fecha_accidente", fecha_ddmmyyyy("fecha_accidente"))
    .withColumn("fecha_nacimiento", fecha_nacimiento("fecha_nacimiento"))
    .withColumn("anio_accidente", F.expr("try_cast(anio_accidente AS INT)"))
    .withColumn("mes_accidente", F.expr("try_cast(mes_accidente AS INT)"))
    .withColumn("dia_accidente", F.expr("try_cast(dia_accidente AS INT)"))
    .withColumn("edad", F.expr("try_cast(edad AS INT)"))
    .withColumn("coordenada_x", decimal_con_coma("coordenada_x"))
    .withColumn("coordenada_y", decimal_con_coma("coordenada_y"))
    .withColumn("croquis", limpiar_texto("croquis"))
    .withColumn("hora_accidente", limpiar_texto("hora_accidente"))
    .withColumn("direccion", limpiar_texto("direccion"))
    .withColumn("descripcion_victima", F.upper(limpiar_texto("descripcion_victima")))
    .withColumn("gravedad", F.upper(limpiar_texto("gravedad")))
    .withColumn("id_usuario", limpiar_texto("id_usuario"))
    .withColumn("sexo", F.upper(limpiar_texto("sexo")))
    .withColumn("placa", F.upper(limpiar_texto("placa")))
    .withColumn("clase_accidente", F.upper(limpiar_texto("clase_accidente")))
    .withColumn("clase_vehiculo", F.upper(limpiar_texto("clase_vehiculo")))
    .withColumn("servicio", F.upper(limpiar_texto("servicio")))
    .withColumn("_silver_transformation_timestamp", F.current_timestamp())
)

display(df_victimas_silver.limit(10))
df_victimas_silver.printSchema()


## 7. Escritura de tablas Silver

Se utiliza `overwrite` porque el proyecto trabaja con un snapshot histórico estático de los CSV. En una ingestión incremental real se evaluaría `MERGE`, `append` u otra estrategia.


In [ ]:
tablas_silver = {
    "accidentes": df_accidentes_silver,
    "causas": df_causas_silver,
    "vehiculos": df_vehiculos_silver,
    "victimas": df_victimas_silver
}

for nombre_tabla, df in tablas_silver.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{nombre_tabla}"
        )
    )

    print(f"[OK] {PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{nombre_tabla}")


## 8. Validación de conteos Bronze vs Silver

Las transformaciones Silver no deben cambiar el número de filas de ninguna fuente.


In [ ]:
nombres_tablas = ["accidentes", "causas", "vehiculos", "victimas"]

for tabla in nombres_tablas:
    bronze_count = spark.table(
        f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.{tabla}"
    ).count()

    silver_count = spark.table(
        f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{tabla}"
    ).count()

    estado = "OK" if bronze_count == silver_count else "REVISAR"

    print(
        f"{tabla:10} | Bronze: {bronze_count:>6,} | "
        f"Silver: {silver_count:>6,} | {estado}"
    )


## 9. Validación de claves y granularidad

Se revisa el número de accidentes distintos y la ausencia de claves nulas.

Esperado:

- accidentes: 23,212 `croquis` distintos;
- causas: 23,212 `croquis` distintos;
- vehículos: 23,215 `croquis` distintos;
- víctimas: 7,758 `croquis` distintos.


In [ ]:
for tabla in nombres_tablas:
    df = spark.table(
        f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{tabla}"
    )

    resumen = df.agg(
        F.count("*").alias("filas"),
        F.countDistinct("croquis").alias("croquis_distintos"),
        F.sum(
            F.when(F.col("croquis").isNull(), 1).otherwise(0)
        ).alias("croquis_null")
    ).collect()[0]

    print(
        f"{tabla:10} | filas={resumen['filas']:,} | "
        f"croquis_distintos={resumen['croquis_distintos']:,} | "
        f"croquis_null={resumen['croquis_null']:,}"
    )


## 10. Validación de conversiones críticas

Si una columna fuente era no nula y después del tipado queda `NULL`, significa que algún valor no pudo convertirse y debe investigarse.


In [ ]:
validaciones_conversion = [
    ("accidentes", "fecha_accidente", "fecha_accidente"),
    ("accidentes", "nro_heridos", "nro_heridos"),
    ("accidentes", "numero_muertos", "numero_muertos"),
    ("accidentes", "coordenada_x", "coordenada_x"),
    ("accidentes", "coordenada_y", "coordenada_y"),
    ("vehiculos", "fecha_accidente", "fecha_accidente"),
    ("vehiculos", "fecha_radica", "fecha_radica"),
    ("vehiculos", "coordenada_x", "coordenada_x"),
    ("vehiculos", "coordenada_y", "coordenada_y"),
    ("victimas", "fecha_accidente", "fecha_accidente"),
    ("victimas", "fecha_nacimiento", "fecha_nacimiento"),
    ("victimas", "edad", "edad"),
    ("victimas", "coordenada_x", "coordenada_x"),
    ("victimas", "coordenada_y", "coordenada_y")
]

for tabla, columna_bronze, columna_silver in validaciones_conversion:
    nulos_bronze = (
        spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_BRONZE}.{tabla}")
        .filter(
            F.col(columna_bronze).isNull()
            | (F.trim(F.col(columna_bronze).cast("string")) == "")
        )
        .count()
    )

    nulos_silver = (
        spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{tabla}")
        .filter(F.col(columna_silver).isNull())
        .count()
    )

    nuevos_nulos = nulos_silver - nulos_bronze
    estado = "OK" if nuevos_nulos == 0 else "REVISAR"

    print(
        f"{tabla}.{columna_silver:18} | "
        f"nulos Bronze={nulos_bronze:>6,} | "
        f"nulos Silver={nulos_silver:>6,} | "
        f"nuevos por conversión={nuevos_nulos:>6,} | {estado}"
    )


## 11. Integridad referencial respecto a accidentes

Se comprueba si los `croquis` presentes en causas, vehículos y víctimas existen también en `silver.accidentes`.

Los registros huérfanos no se eliminan: se reportan como hallazgo de calidad.


In [ ]:
accidentes_keys = (
    spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes")
    .select("croquis")
    .distinct()
)

for tabla in ["causas", "vehiculos", "victimas"]:
    keys = (
        spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.{tabla}")
        .select("croquis")
        .distinct()
    )

    huerfanos = keys.join(
        accidentes_keys,
        on="croquis",
        how="left_anti"
    )

    total_huerfanos = huerfanos.count()

    print(
        f"{tabla:10} | croquis sin accidente asociado: "
        f"{total_huerfanos:,}"
    )

    if total_huerfanos > 0:
        display(huerfanos.orderBy("croquis"))


## 12. Consistencia entre accidentes y víctimas

Se valida que la cantidad de víctimas registrada por accidente sea consistente con la suma de heridos y muertos reportada en `silver.accidentes`.

Para esta validación, los `NULL` de heridos o muertos se interpretan como cero **solo dentro del cálculo de control**, sin modificar los valores almacenados en Silver.


In [ ]:
victimas_por_accidente = (
    spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.victimas")
    .groupBy("croquis")
    .agg(F.count("*").alias("cantidad_victimas"))
)

control_victimas = (
    spark.table(f"{PROY_CATALOG}.{PROY_SCHEMA_SILVER}.accidentes")
    .select(
        "croquis",
        (
            F.coalesce(F.col("nro_heridos"), F.lit(0))
            + F.coalesce(F.col("numero_muertos"), F.lit(0))
        ).alias("victimas_reportadas")
    )
    .join(victimas_por_accidente, on="croquis", how="left")
    .fillna({"cantidad_victimas": 0})
    .withColumn(
        "coincide",
        F.col("victimas_reportadas") == F.col("cantidad_victimas")
    )
)

display(control_victimas.groupBy("coincide").count())

display(
    control_victimas
    .filter(~F.col("coincide"))
    .limit(20)
)


## 13. Resultado

Al finalizar correctamente este notebook deben existir:

- `silver.accidentes`
- `silver.causas`
- `silver.vehiculos`
- `silver.victimas`

Estas tablas serán la base para las siguientes etapas del proyecto: Delta Lake, controles adicionales, modelo dimensional Gold y dataset de Machine Learning.
